In [1]:
import gymnasium
import highway_env
import time
import os
import numpy as np
from Functions.Graphs import *
from highway_env.vehicle.objects import Obstacle # Import a static object to anchor the camera

c:\Users\Claudio\Desktop\Python\ControlApplications\venv\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [ ]:
sim_rate = 50
# 1. Instantiate the environment
env = gymnasium.make(
    "racetrack-v0",
    render_mode="human",
    config={
        "screen_width": 900,
        "screen_height": 900,
        "action": {
            "type": "DiscreteMetaAction",
            "target_speeds": [0, 1,2,3,4,5,6,7,8,9, 10], # Correctly nested
        },
        "other_vehicles": 0,
        "controlled_vehicles": 1,
        "duration": 100,
        "simulation_frequency": sim_rate,
        "policy_frequency": 10,
        'show_trajectories': True,
        "centering_position": [0.15, 0.5], # Uses the correct configuration key name
        "scaling": 5,
    }
)

obs, info = env.reset()
camera_anchor = Obstacle(env.unwrapped.road, position=[0, 0])
env.unwrapped.viewer.observer_vehicle = camera_anchor
# Use the Action Type's built-in helper to find indices
actions = env.unwrapped.action_type.actions_indexes
vehicle = env.unwrapped.vehicle
vehicle.speed = 0.0
laps = 0
previous_node = vehicle.lane_index[0]
phase = 'ACCELERATING'

print("Simulation started. Phase: ACCELERATING")

acc_ref = np.array([])
acc_cur = np.array([])
spd_ref = np.array([])
spd_cur = np.array([])
angle = np.array([])
k_gain = np.array([])
while phase != 'FINISHED':
    current_node = vehicle.lane_index[0]
    current_speed = vehicle.speed
    target_speed = vehicle.target_speed

    kp_a = getattr(vehicle, 'KP_A', 0.6)
    reference_acceleration = kp_a * (target_speed - current_speed)

    current_acceleration = vehicle.action.get('acceleration', 0.0)
    current_steering_rad = vehicle.action.get('steering', 0.0)
    current_steering_deg = np.degrees(current_steering_rad)

    acc_ref = np.append(acc_ref,reference_acceleration)
    acc_cur = np.append(acc_cur,current_acceleration)
    spd_ref = np.append(spd_ref,target_speed)
    spd_cur = np.append(spd_cur,current_speed)
    k_gain = np.append(k_gain,kp_a)
    angle = np.append(angle,current_steering_deg)

    print('Phase:',phase, 'Current Speed:',current_speed, 'Reference Acceleration:',reference_acceleration, 'Current Acceleration:',current_acceleration, 'Stirring Angle:',current_steering_deg)
    # --- LAP TRACKING LOGIC ---
    if previous_node == "i" and current_node == "a":
        laps += 1
        print(f"Lap boundary crossed! Total laps: {laps}")
    
    # --- PHASE & SPEED CONTROL LOGIC ---
    
    # Note: We no longer manually set vehicle.target_speed.
    # We let the environment's action handler manage it via FASTER/SLOWER.

    if phase == 'ACCELERATING':
        if current_speed < 9.5:
            action = actions["IDLE"]
        else:
            phase = 'CRUISING'
            print("Target speed reached. Phase: CRUISING")
            action = actions["IDLE"]

    elif phase == 'CRUISING':
        if laps >= 1:
            phase = 'BRAKING'
            print("Approaching finish line. Phase: BRAKING")
            action = actions["SLOWER"]
        else:
            # Simple cruise maintenance
            if current_speed < 9.0:
                action = actions["FASTER"]
            elif current_speed > 11.0:
                action = actions["SLOWER"]
            else:
                action = actions["IDLE"]

    elif phase == 'BRAKING':
        if current_speed > 0.1:
            action = actions["SLOWER"]
        else:
            action = actions["IDLE"]
            # Ensure we are at the finish line node 'a' and fully stopped
            if current_node == "a" and current_speed <= 0.01:
                phase = 'FINISHED'
                print(f"Vehicle stopped successfully. Final Speed: {current_speed:.2f} m/s")

    # Execute action
    obs, reward, done, truncated, info = env.step(action)
    env.render()
    
    previous_node = current_node

    if done or truncated:
        print("Vehicle crashed or execution timed out.")
        break

print("Simulation finished successfully.")
env.close()


In [ ]:
import gymnasium as gym
import numpy as np
from highway_env.vehicle.objects import Obstacle
sim_rate = 100
pol_rate = 25
sim_per = (sim_rate/pol_rate)
# 1. Instantiate the environment
env = gym.make(
    "racetrack-v0",
    render_mode="human",
    config={
        "screen_width": 900,
        "screen_height": 900,
        "action": {
            "type": "DiscreteMetaAction",
            # Increased target speeds to allow for a maximum speed of 20 m/s
            "target_speeds": [0, 5, 10, 15, 20, 25],
        },
        "other_vehicles": 0,
        "controlled_vehicles": 1,
        "duration": 100,
        "simulation_frequency": sim_rate,
        "policy_frequency": pol_rate,
        'show_trajectories': True,
        "centering_position": [0.15, 0.5],
        "scaling": 5,
    }
)

obs, info = env.reset()
camera_anchor = Obstacle(env.unwrapped.road, position=[0, 0])
env.unwrapped.viewer.observer_vehicle = camera_anchor

#obs, info = env.reset()
# Make the camera follow the main vehicle for better observation
#env.unwrapped.viewer.observer_vehicle = env.unwrapped.vehicle
# Use the Action Type's built-in helper to find indices
actions = env.unwrapped.action_type.actions_indexes
vehicle = env.unwrapped.vehicle
vehicle.speed = 0.0
vehicle.position[0],vehicle.position[1] = 72, 0
print(vehicle.position[0],vehicle.position[1])
laps = 0
previous_node = vehicle.lane_index[0]
phase = 'ACCELERATING'

print("Simulation started. Phase: ACCELERATING")

acc_ref = np.array([])
acc_cur = np.array([])
spd_ref = np.array([])
spd_cur = np.array([])
angle = np.array([])
k_gain = np.array([])
x_coords = np.array([])
y_coords = np.array([])
psi_history = []
beta_history = []

MAX_SPEED = 20.0 # Define the desired maximum speed


while phase != 'FINISHED':
    current_node = vehicle.lane_index[0]
    current_speed = vehicle.speed
    target_speed = vehicle.target_speed
    current_x = vehicle.position[0]
    current_y = vehicle.position[1]
    current_psi_rad = vehicle.heading
    current_psi_deg = np.degrees(current_psi_rad)
    current_beta_rad = np.arctan(0.5 * np.tan(current_steering_rad))
    current_beta_deg = np.degrees(current_beta_rad)

    kp_a = getattr(vehicle, 'KP_A', 0.6)
    reference_acceleration = kp_a * (target_speed - current_speed)

    current_acceleration = vehicle.action.get('acceleration', 0.0)
    current_steering_rad = vehicle.action.get('steering', 0.0)
    current_steering_deg = np.degrees(current_steering_rad)

    psi_history.append(current_psi_deg)
    beta_history.append(current_beta_deg)
    x_coords = np.append(x_coords, current_x)
    y_coords = np.append(y_coords, current_y)
    acc_ref = np.append(acc_ref,reference_acceleration)
    acc_cur = np.append(acc_cur,current_acceleration)
    spd_ref = np.append(spd_ref,target_speed)
    spd_cur = np.append(spd_cur,current_speed)
    k_gain = np.append(k_gain,kp_a)
    angle = np.append(angle,current_steering_deg)



    #print('Phase:',phase, 'Current Speed:',f'{current_speed:.2f}', 'Reference Acceleration:',f'{reference_acceleration:.2f}', 'Current Acceleration:',f'{current_acceleration:.2f}', 'Stirring Angle:',f'{current_steering_deg:.2f}')
    # --- LAP TRACKING LOGIC ---
    if previous_node == "i" and current_node == "a":
        laps += 1
        print(f"Lap boundary crossed! Total laps: {laps}")
    
    # --- PHASE & SPEED CONTROL LOGIC ---
    
    # Note: We no longer manually set vehicle.target_speed.
    # We let the environment's action handler manage it via FASTER/SLOWER.

    if phase == 'ACCELERATING':
        # If the vehicle's current target speed is less than MAX_SPEED, try to increase it.
        # The `DiscreteMetaAction` will handle incrementing `vehicle.target_speed`
        # through the `target_speeds` list.
        if vehicle.target_speed < MAX_SPEED:
            action = actions["FASTER"]
        else:
            # Once the target speed is MAX_SPEED, transition to cruising.
            # The vehicle will then try to reach and maintain this speed.
            phase = 'CRUISING'
            print(f"Target speed {MAX_SPEED} m/s set. Phase: CRUISING")
            action = actions["IDLE"] # Maintain the target speed

    elif phase == 'CRUISING':
        # Maintain speed around MAX_SPEED.
        # The `DiscreteMetaAction` will try to keep `vehicle.target_speed` at MAX_SPEED.
        # We use IDLE to tell the meta-action to not change the target speed.
        if laps >= 1: # Assuming 1 lap is enough to trigger braking
            phase = 'BRAKING'
            print('BRAKING', vehicle.position[0],vehicle.position[1])
            print("Approaching finish line. Phase: BRAKING")
            action = actions["SLOWER"] # Start braking by reducing target speed
        else:
            # If current speed deviates significantly from MAX_SPEED, adjust target speed.
            # This is a fine-tuning for the meta-action.
            if vehicle.speed < MAX_SPEED - 1.0:
                action = actions["FASTER"]
            elif vehicle.speed > MAX_SPEED + 1.0:
                action = actions["SLOWER"]
            else:
                action = actions["IDLE"] # Maintain current target speed

    elif phase == 'BRAKING':
        # If the vehicle's current target speed is greater than 0, try to decrease it.
        # The `DiscreteMetaAction` will handle decrementing `vehicle.target_speed`
        # through the `target_speeds` list.
        if vehicle.target_speed > 0:
            action = actions["SLOWER"]
        else:
            # Once the target speed is 0, wait for the vehicle to actually stop
            # and be at the finish line.
            action = actions["IDLE"] # Maintain target speed at 0
            if current_node == "a" and current_speed <= 0.1:
                phase = 'FINISHED'
                print('FINISHED', vehicle.position[0],vehicle.position[1])
                print(f"Vehicle stopped successfully. Final Speed: {current_speed:.2f} m/s")
            else:
                # If not at finish line 'a' yet, or not fully stopped, keep trying to slow down
                # (though target_speed is already 0, this ensures the action is still 'SLOWER'
                # if the vehicle hasn't reached 0 yet, or 'IDLE' if it has).
                action = actions["SLOWER"] if current_speed > 0.01 else actions["IDLE"]

    # Execute action
    obs, reward, done, truncated, info = env.step(action)
    env.render()
    
    previous_node = current_node

    if done or truncated:
        print("Vehicle crashed or execution timed out.")
        break

print("Simulation finished successfully.")
env.close()
sim_duration = len(angle)/sim_per
time_indx = np.linspace(0,sim_duration,len(angle))


72.0 0.0
Simulation started. Phase: ACCELERATING
Target speed 20.0 m/s set. Phase: CRUISING


KeyboardInterrupt: 

: 

In [84]:
df = pd.DataFrame()
df['time'] = time_indx
df['x_coords'] = x_coords
df['y_coords'] = y_coords  
df['angle'] =  angle
df['psi'] =  psi_history
df['beta'] =  beta_history
df['speed'] =  spd_cur

out_dir = r'DyntheticDataset/'
os.makedirs(out_dir,exist_ok=True)
out_file = out_dir + 'RaceTrack.csv'
df.to_csv(out_file,index=False)

In [44]:
PlotTwoScalesPLY(y1=x_coords,y2=y_coords)

In [45]:
PlotSeriesPLY(ySeries=[angle])

In [46]:
PlotSeriesPLY(ySeries=[acc_ref,acc_cur])

In [47]:
PlotSeriesPLY(ySeries=[spd_ref,spd_cur])